In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('../..'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc

plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 15,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})


In [ ]:
folder = "/home/liangmj/repo/LBM-Program/amrlbm/bin"

fileVec = [

    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_moving_cylinder_channel/fixed_frame_2d_bgk_avgMacroEq_extrapNeq/forcetorque/forceCylinder/forcetorque0000.txt",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_moving_cylinder_channel/fixed_frame_2d_bgk_eq/forcetorque/forceCylinder/forcetorque0000.txt",

    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_moving_cylinder_channel/fixed_frame_3d_cml_avgMacroEq_extrapNeq/forcetorque/forceCylinder/forcetorque0000.txt",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_moving_cylinder_channel/fixed_frame_3d_cml_eq/forcetorque/forceCylinder/forcetorque0000.txt",

    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_moving_cylinder_channel/fixed_frame_3d_bgk_eq/forcetorque/forceCylinder/forcetorque0000.txt",

    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_moving_cylinder_channel/moving_frame_2d_bgk/forcetorque/forceCylinder/forcetorque0000.txt",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_moving_cylinder_channel/moving_frame_3d_cml/forcetorque/forceCylinder/forcetorque0000.txt",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_moving_cylinder_channel/moving_frame_3d_bgk/forcetorque/forceCylinder/forcetorque0000.txt",

]

rho0 = 1.0
rhoPhy = 1.204
startStep = 0

rVec = [
    24*0.5,
    24*0.5,
    24*0.5,
    24*0.5,
]


colorVec = [
    # "#CC9D02",
    "#D33737",
    # "#001AFF",


    # "#51C8EC",
    "#05A361",
    # "#B700FF",

]

linestyleVec = [
    "--",
    "-.",
    ":",
    "-",
]

labelVec = [
    # r"D2Q9 BGK FIXED frame $f^{eq}(\rho^*, u^*)+f^{neq}$",
    # r"D2Q9 BGK FIXED frame $f^{eq}(\rho_0, u_c)$",

    # r"D3Q27 Cumulant FIXED frame $f^{eq}(\rho^*, u^*)+f^{neq}$",
    r"D3Q27 Cumulant FIXED frame $f^{eq}(\rho_0, u_c)$",

    # r"D3Q19 BGK FIXED frame $f^{eq}(\rho^*, u^*)+f^{neq}$",
    # r"D3Q19 BGK FIXED frame $f^{eq}(\rho_0, u_c)$",


    # "D2Q9 BGK MOVING frame",
    "D3Q27 Cumulant MOVING frame",
    # "D3Q19 BGK MOVING frame",


]

gamma = 1.0
dxVec = [1.0 / (2 * r) for r in rVec]
ucVec = [uc(dx, rhoPhy, gamma) for dx in dxVec]



In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

In [ ]:


stepVec = []
timeVec = []
forceVec = [[], [], []]
torqueVec = [[], [], []]
for case in range(0, len(fileVec)):
    stepCol = read_col_probe(fileVec[case], 0, 2)
    fx = read_col_probe(fileVec[case], 2, 2)
    fy = read_col_probe(fileVec[case], 3, 2)
    fz = read_col_probe(fileVec[case], 4, 2)
    tx = read_col_probe(fileVec[case], 5, 2)
    ty = read_col_probe(fileVec[case], 6, 2)
    tz = read_col_probe(fileVec[case], 7, 2)
    
    idxStart = int(np.abs(stepCol - startStep).argmin())
    stepVec.append(stepCol[idxStart:])
    timeVec.append(stepCol[idxStart:] * ucVec[case].factor_time)
    forceVec[0].append(fx[idxStart:])
    forceVec[1].append(fy[idxStart:])
    forceVec[2].append(fz[idxStart:])
    torqueVec[0].append(tx[idxStart:])
    torqueVec[1].append(ty[idxStart:])
    torqueVec[2].append(tz[idxStart:])


In [ ]:

figF, axF = plt.subplots(3, 2, figsize=(16, 8), facecolor='w', edgecolor='w', sharex=True)
alpha = 1
for case in range(0, len(fileVec)):
    for i in [0, 1, 2]:
        axF[i][0].plot(stepVec[case], forceVec[i][case], c=colorVec[case], lw=1.5, label=labelVec[case], alpha=alpha)
        axF[i][1].plot(stepVec[case], torqueVec[i][case], c=colorVec[case], lw=1.5, label=labelVec[case], alpha=alpha)

axF[0][0].set_ylabel('Force-x')
axF[1][0].set_ylabel('Force-y')
axF[2][0].set_ylabel('Force-z')
axF[2][0].set_xlabel('Step')

axF[0][1].set_ylabel('Torque-x')
axF[1][1].set_ylabel('Torque-y')
axF[2][1].set_ylabel('Torque-z')
axF[2][1].set_xlabel('Step')

# axF[0][0].set_xlim(10000, 20000)
axF[0][0].set_ylim(-0.07, 0.07)
# axF[1][0].set_ylim(-0.05, 0.05)

# axF[0][0].set_xlim(2000, 2500)
# axF[0][0].set_ylim(-0.25, 0.25)

# axF[2][1].set_xlim(0, 5000)
# axF[2][1].set_ylim(-0.2, 0.01)

axF[0][0].legend(loc='best', frameon=False, fontsize=12)
figF.tight_layout()

In [ ]:
figF1, axF1 = plt.subplots(1, 1, figsize=(12, 5), facecolor='w', edgecolor='w', sharex=True)
alpha = 1.0
for case in range(0, len(fileVec)):
    axF1.plot(stepVec[case], forceVec[0][case], c=colorVec[case], lw=1.5, label=labelVec[case], linestyle='-.', alpha=alpha)
    axF1.plot(stepVec[case], forceVec[1][case], c=colorVec[case], lw=1.5, alpha=alpha)
# axF1.set_xlim(14000, 19000)
axF1.set_xlim(12600, 17600)
# axF1.set_xlim(500, 3000)
axF1.set_ylim(-0.05, 0.02)
# axF1.set_ylim(-0.1, 0.05)
axF1.set_xlabel(r"$t$")
axF1.set_ylabel(r"$F_{x}(t), F_{y}(t)$")
axF1.legend(loc='upper right', bbox_to_anchor=(1.0, 1.2), frameon=False)
axF1.tick_params(width=2.0, axis='both', direction='in')
for spine in axF1.spines.values():
    spine.set_linewidth(2.0)